# Benchmark toàn pipeline — arXiv + Semantic Scholar

Notebook dành cho **benchmark**, không lưu corpus TeX hoặc response metadata/reference. Chỉ giữ JSON số đo, trạng thái và checkpoint.
Input Version IDs → discovery/metadata lookup → tải source + xử lý file → S2 → báo cáo.
Chạy benchmark cuối trên **Google Colab, CPU-only** (Runtime → Change runtime type → CPU).

Giữ input cũ: `IDS_TXT_DIR`, `START_IDX`, `END_IDX`; mỗi dòng một ID có version, khoảng dòng `[start, end)`.
S2 mặc định dùng unique base IDs từ cùng slice. `REF_IDS_PATH` là file ID gốc tùy chọn chọn tập con; không crawl ID ngoài slice.
Không tự tải Version ngoài input. `done.json` xác nhận đủ source trong manifest; `pipeline_complete` nói benchmark đủ các Version và S2, được discovery xác minh. **Không có nghĩa đã xuất dataset theo schema.**

Chuẩn bị: upload file ID; dùng Python **3.12+** và packages `requests`, `aiohttp`, `psutil`.
Nếu clone repo, có thể chạy `%pip install -r src/requirements-crawling.txt` trong Colab; hoặc `%pip install requests aiohttp psutil` nếu chỉ upload notebook.
Notebook độc lập, không cần SQLite/module repo. Điền `CONTACT_EMAIL` và mảng `S2_API_KEYS` trong cell cấu hình; mỗi phần tử một worker async. Không in/ghi key vào báo cáo. Bản bàn giao không có key thật; sau khi tự điền, giữ bản notebook đó ở nơi riêng.

**Tách mạng và file:** một producer tải source tuần tự, một worker xử lý file đồng thời, giữa hai bên là queue tối đa `FILE_QUEUE_SIZE` archive. Archive chỉ đưa vào queue sau khi tải xong. Bounded queue hạn chế số archive chờ, không phải giới hạn bytes; peak disk vẫn được đo trước khi dọn file.
Source được giải nén thật để đo trước/sau lọc `.tex/.bib`, rồi bỏ. S2 đếm theo từng page rồi bỏ payload, không tích lũy raw references.
`CLEANUP_MODE` giữ hai lựa chọn `immediate`/`batch` như benchmark cũ.

Cold benchmark dùng `OUTPUT_ROOT` mới. Resume cùng input/output chỉ dùng lại số đo/status thành công, không kiểm tra corpus vì payload đã được dọn có chủ ý. Thời gian/RAM/HTTP phiên resume không đại diện cold benchmark.

Nguồn: [đề Milestone 1](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/), [arXiv API](https://info.arxiv.org/help/api/user-manual.html), [S2 Graph API](https://api.semanticscholar.org/api-docs/graph).
Xem kế hoạch và checklist metric trong `docs/crawling-pipeline-plan.md`.


## 1. Input và cấu hình
Sửa cell này, sau đó chạy lần lượt. `S2_API_KEYS` là mảng Python, mỗi phần tử một worker.

In [ ]:
import os
import tempfile
from pathlib import Path

# Giữ input: mỗi dòng một ID có version; chỉ số dòng bắt đầu từ 0.
IDS_TXT_DIR = Path("/content/arxiv_id.txt")
START_IDX = 0
END_IDX = 10  # [START_IDX, END_IDX): không gồm dòng END_IDX.
REF_IDS_PATH = None  # Hoặc Path("/content/arxiv_id_no_version.txt"). Chỉ lấy ID thuộc slice.
OUTPUT_ROOT = Path("/content/crawling-pipeline")
TEMP_DIR = Path(tempfile.gettempdir()) / "crawling-pipeline-temp"
# Kaggle: IDS_TXT_DIR = Path("/kaggle/input/TEN_DATASET/arxiv_id.txt")
# Kaggle: OUTPUT_ROOT = Path("/kaggle/working/crawling-pipeline")
# Local từ root repo: IDS_TXT_DIR = Path("src/data/arxiv_id.txt")
# Local: OUTPUT_ROOT = Path("data/crawling-pipeline")

DISCOVERY_MODE = "live"  # "live": resolve metadata; "precomputed_input": không đo discovery ngoài notebook.
DISCOVERY_BATCH_SIZE = 50
RESUME = True  # Đọc checkpoint cùng input; cold run dùng OUTPUT_ROOT mới.
CLEANUP_MODE = "immediate"  # "immediate" | "batch": dọn toàn bộ payload sau khi đo.
BATCH_SIZE = 100
FILE_QUEUE_SIZE = 2  # Tối đa 2 archive đang chờ xử lý; một downloader + một file worker.

MIN_REQUEST_INTERVAL = 3.0
REQUEST_TIMEOUT = (10, 120)
RETRY_ROUND_DELAY = 30
MAX_RETRY_DELAY = 300
MAX_RETRY_ROUNDS = 10  # Số vòng thử lại sau lượt đầu, áp dụng riêng mỗi stage.
CONTACT_EMAIL = os.environ.get("CONTACT_EMAIL", "")  # Hoặc điền email liên hệ ở đây.
USER_AGENT = f"arxiv-reference-matching/0.1 (HCMUS intro-ds; mailto:{CONTACT_EMAIL})"

# Một phần tử = một worker. Điền trực tiếp key ở đây, không in mảng này.
S2_API_KEYS = [
    # "PASTE_KEY_1",
    # "PASTE_KEY_2",
]
S2_BATCH_SIZE = 50
S2_REQUEST_INTERVAL = 1.0
S2_REQUEST_TIMEOUT = 60

MEMBER = "minh"
COLD_RUN = True  # Giá trị thực được suy ra từ checkpoint khi chạy.
BENCHMARK_INTERVAL = 1.0
DISK_INTERVAL = 5.0

METADATA_OUTPUT_DIR = OUTPUT_ROOT / "metadata.json"  # Giữ lịch sử id/path/state của source.
STATUS_OUTPUT_PATH = OUTPUT_ROOT / "status.json"
DONE_OUTPUT_PATH = OUTPUT_ROOT / "done.json"  # Hoàn tất SOURCE, không có nghĩa S2 đã thành công.
SKIPPED_OUTPUT_PATH = OUTPUT_ROOT / "skipped_ids.json"
PAPERS_STATS_OUTPUT_PATH = OUTPUT_ROOT / "papers_stats.json"
SUMMARY_OUTPUT_PATH = OUTPUT_ROOT / "summary.json"
BENCHMARK_OUTPUT_PATH = OUTPUT_ROOT / "benchmark.json"
BENCHMARK_SAMPLES_OUTPUT_PATH = OUTPUT_ROOT / "benchmark_samples.json"
CLAIM_OUTPUT_PATH = OUTPUT_ROOT / "benchmark.claim.json"
RUN_OUTPUT_PATH = OUTPUT_ROOT / "run.json"
DISCOVERY_STATUS_PATH = OUTPUT_ROOT / "discovery_status.json"
S2_STATUS_PATH = OUTPUT_ROOT / "s2_status.json"
CHECKPOINT_PATH = OUTPUT_ROOT / "checkpoint.json"


## 2. Imports, manifest và JSON
Đọc đúng slice theo dòng vật lý, bỏ dòng trống, không gửi lại tác vụ trùng ID. Giữ ID cũ dạng `hep-th/9901001`.

In [ ]:
import asyncio
from collections import Counter
from datetime import datetime, timezone
import gzip
import hashlib
from importlib.metadata import version as package_version
import json
import platform
import queue
import re
import shutil
import statistics
import sys
import tarfile
import threading
import time
from urllib.parse import quote
from xml.etree import ElementTree as ET
import zipfile
import zlib

import aiohttp
import psutil
import requests

MB = 1024 * 1024
TERMINAL_SOURCE = {"ok", "empty"}
NS = {"atom": "http://www.w3.org/2005/Atom", "arxiv": "http://arxiv.org/schemas/atom"}


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def read_json(path, default):
    path = Path(path)
    # Checkpoint hỏng phải báo lỗi, không âm thầm ghi đè bằng dict rỗng.
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default


def version_output(version_id, save_dir=TEMP_DIR):
    base_id, suffix = version_id.rsplit("v", 1)
    if not suffix.isdigit() or int(suffix) < 1:
        raise ValueError(f"ID cần version v1 trở lên: {version_id}")
    if not re.fullmatch(r"(?:\d{4}\.\d{4,5}|[a-zA-Z-]+(?:\.[A-Z]{2})?/\d{7})", base_id):
        raise ValueError(f"arXiv ID không hợp lệ: {version_id}")
    key = base_id.replace("/", "-").replace(".", "-")
    return base_id, int(suffix), Path(save_dir) / key / f"v{int(suffix)}"


def get_id_list():
    lines = Path(IDS_TXT_DIR).read_text(encoding="utf-8").splitlines()
    return [line.strip() for line in lines[START_IDX:END_IDX] if line.strip()]


def load_manifest():
    if not 0 <= START_IDX <= END_IDX:
        raise ValueError("Cần 0 <= START_IDX <= END_IDX")
    lines = Path(IDS_TXT_DIR).read_text(encoding="utf-8").splitlines()
    expected = {}
    for value in dict.fromkeys(line.strip() for line in lines if line.strip()):
        base_id, number, _ = version_output(value)
        expected.setdefault(base_id, {})[number] = value
    input_ids = get_id_list()
    selected = list(dict.fromkeys(input_ids))
    if not selected:
        raise ValueError("Lát cắt ID rỗng")
    base_ids = list(dict.fromkeys(version_output(value)[0] for value in selected))
    reference_ids = base_ids
    extra_ref_ids = []
    if REF_IDS_PATH is not None:
        requested = list(dict.fromkeys(line.strip() for line in Path(REF_IDS_PATH).read_text().splitlines() if line.strip()))
        reference_ids = [value for value in requested if value in base_ids]
        extra_ref_ids = [value for value in requested if value not in base_ids]
    identity = {
        "ids_sha256": hashlib.sha256(Path(IDS_TXT_DIR).read_bytes()).hexdigest(),
        "selected_ids": selected, "reference_ids": reference_ids,
        "discovery_mode": DISCOVERY_MODE, "benchmark_only": True,
    }
    return {
        "selected_ids": selected, "base_ids": base_ids, "reference_ids": reference_ids,
        "expected": expected, "duplicate_selected_lines": len(input_ids) - len(selected),
        "reference_ids_outside_slice": extra_ref_ids, "identity": identity,
    }


def new_http_stats():
    return {"requests": 0, "retries": 0, "retry_rounds": 0, "http_429": 0,
            "http_5xx": 0, "backoff_seconds": 0.0, "errors_by_type": {}}


def count_error(stats, error, status=None):
    key = str(status) if status is not None else type(error).__name__
    stats["errors_by_type"][key] = stats["errors_by_type"].get(key, 0) + 1
    stats["http_429"] += status == 429
    stats["http_5xx"] += status is not None and 500 <= status < 600


def retry_delay(round_number, retry_after=0):
    return max(min(RETRY_ROUND_DELAY * 2 ** (round_number - 1), MAX_RETRY_DELAY), retry_after)


def retry_after_seconds(headers):
    value = (headers or {}).get("Retry-After", "")
    try:
        return max(0.0, float(value))
    except (ValueError, TypeError):
        # HTTP-date cũng hợp lệ, không chỉ số giây.
        from email.utils import parsedate_to_datetime
        try:
            return max(0.0, parsedate_to_datetime(value).timestamp() - time.time())
        except (ValueError, TypeError, OverflowError):
            return 0.0


def average(values):
    values = [value for value in values if value is not None]
    return round(statistics.mean(values), 6) if values else None


def ratio(part, total):
    return round(part / total, 6) if total else None


def segments_for(base_ids):
    segments = []
    modern = sorted((value.split(".")[0], int(value.split(".")[1])) for value in base_ids if "/" not in value)
    for prefix, number in modern:
        if segments and segments[-1]["prefix"] == prefix and segments[-1]["end"] == number - 1:
            segments[-1]["end"] = number
        else:
            segments.append({"prefix": prefix, "start": number, "end": number})
    return segments


## 3. Đo toàn hệ thống
Monitor bao cả discovery, source, S2 và export lần đầu. RSS gồm kernel hiện tại và tiến trình con, kể cả baseline notebook. RAM trung bình dùng tích phân hình thang theo timestamp; disk đo các root không đếm trùng. Peak là giá trị quan sát qua lấy mẫu, không phải bảo đảm bắt được mọi spike ngắn hơn chu kỳ lấy mẫu.

In [ ]:
def dir_size(path):
    total = 0
    for root, _dirs, files in os.walk(path):
        for name in files:
            try:
                total += os.path.getsize(os.path.join(root, name))
            except OSError:
                pass  # File tạm có thể vừa được dọn bởi worker.
    return total


def tree_rss(process):
    total = process.memory_info().rss
    for child in process.children(recursive=True):
        try:
            total += child.memory_info().rss
        except psutil.Error:
            pass
    return total


def get_machine_info():
    return {
        "platform": platform.platform(), "python": sys.version, "cpus": os.cpu_count(),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
        "output_disk_free_gib": round(shutil.disk_usage(OUTPUT_ROOT).free / 2**30, 2),
        "temp_disk_free_gib": round(shutil.disk_usage(TEMP_DIR).free / 2**30, 2),
        "colab": "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ,
        "kaggle": bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")),
        "gpu_visible": bool(os.environ.get("COLAB_GPU", "0") not in {"", "0"} or Path("/dev/nvidia0").exists()),
        "packages": {name: package_version(name) for name in ("requests", "aiohttp", "psutil")},
    }


class BenchmarkMonitor:
    """RSS toàn cây tiến trình; average theo thời gian, disk gồm output và scratch."""

    def __init__(self, output_dir, temp_dir, interval=1.0, disk_interval=5.0):
        self.output_dir, self.temp_dir = Path(output_dir), Path(temp_dir)
        self.interval, self.disk_interval = interval, disk_interval
        self.samples = []
        self.peak_output_disk_mb = self.peak_temp_disk_mb = self.peak_total_disk_mb = 0.0
        self.last_output_disk_mb = self.last_temp_disk_mb = 0.0
        self._lock = threading.Lock()
        self._stop_event = threading.Event()
        self._proc = psutil.Process(os.getpid())
        self._thread = None
        self.ended_at = self.finished_iso = None

    def disk_sizes(self):
        output = dir_size(self.output_dir)
        scratch = dir_size(self.temp_dir)
        # Nếu một root nằm trong root kia, tổng chỉ đếm một lần.
        a, b = self.output_dir.resolve(), self.temp_dir.resolve()
        total = output if b.is_relative_to(a) else scratch if a.is_relative_to(b) else output + scratch
        return output / MB, scratch / MB, total / MB

    def sample_once(self, force_disk=False):
        with self._lock:
            now = time.monotonic() - self._t0
            if force_disk:
                output, scratch, total = self.disk_sizes()
                self.last_output_disk_mb, self.last_temp_disk_mb = output, scratch
                self.peak_output_disk_mb = max(self.peak_output_disk_mb, output)
                self.peak_temp_disk_mb = max(self.peak_temp_disk_mb, scratch)
                self.peak_total_disk_mb = max(self.peak_total_disk_mb, total)
                self._last_total_disk_mb = total
            self.samples.append({
                "t_seconds": now, "rss_mb": tree_rss(self._proc) / MB,
                "disk_mb": self._last_total_disk_mb,
                "output_disk_mb": self.last_output_disk_mb, "temp_disk_mb": self.last_temp_disk_mb,
            })

    def _loop(self):
        last_disk = time.monotonic()
        while not self._stop_event.wait(self.interval):
            measure_disk = time.monotonic() - last_disk >= self.disk_interval
            self.sample_once(force_disk=measure_disk)
            if measure_disk:
                last_disk = time.monotonic()

    def start(self):
        self._t0 = time.monotonic()
        self.started_at = time.time()
        self.started_iso = datetime.now(timezone.utc).isoformat()
        self.sample_once(force_disk=True)
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread:
            self._thread.join()
        self.sample_once(force_disk=True)
        self.ended_at = time.time()
        self.finished_iso = datetime.now(timezone.utc).isoformat()

    def summary(self, done_at_start=0, done_at_end=0):
        samples = self.samples.copy()
        duration = samples[-1]["t_seconds"] - samples[0]["t_seconds"]
        # Tích phân hình thang: mẫu ép khi cleanup không làm lệch trọng số average.
        area = sum((b["t_seconds"] - a["t_seconds"]) * (a["rss_mb"] + b["rss_mb"]) / 2
                   for a, b in zip(samples, samples[1:]))
        output, scratch, total = self.disk_sizes()
        return {
            "wall_seconds": samples[-1]["t_seconds"], "started_at": self.started_at,
            "ended_at": self.ended_at, "started_iso": self.started_iso, "finished_iso": self.finished_iso,
            "peak_rss_mb": max(sample["rss_mb"] for sample in samples),
            "avg_rss_mb": area / duration if duration > 0 else samples[-1]["rss_mb"],
            "ram_average_method": "time_weighted_trapezoidal_rss_process_tree",
            "baseline_rss_mb": samples[0]["rss_mb"],
            "peak_disk_mb": max(self.peak_total_disk_mb, total),
            "peak_output_disk_mb": max(self.peak_output_disk_mb, output),
            "peak_temp_disk_mb": max(self.peak_temp_disk_mb, scratch),
            "final_disk_mb": output, "final_temp_disk_mb": scratch,
            "n_samples": len(samples), "done_papers_at_start": done_at_start,
            "done_papers_at_end": done_at_end,
        }


## 4. Discovery / metadata lookup
`live` resolve latest Version từ các base ID đã chọn, chỉ lưu ID/version count phục vụ kiểm tra độ phủ; bỏ metadata thô. Không đổi input tải. Thời gian tạo manifest ngoài notebook không đo được. `precomputed_input` bỏ lookup: discovery time `null`, `end_to_end_complete=false`. Notebook này không thu thập ngày revision để xuất bảng `papers`.

In [ ]:
def wait_for_rate_limit(context):
    remaining = MIN_REQUEST_INTERVAL - (time.monotonic() - context["last_arxiv_request"])
    if remaining > 0:
        time.sleep(remaining)
    context["last_arxiv_request"] = time.monotonic()


def arxiv_get(context, url, stage, **kwargs):
    """Discovery và source dùng cùng Session, cùng nhịp request."""
    wait_for_rate_limit(context)
    context["http"][stage]["requests"] += 1
    return context["arxiv_session"].get(url, timeout=REQUEST_TIMEOUT, **kwargs)


def parse_metadata(xml_bytes, requested_ids):
    root = ET.fromstring(xml_bytes)
    found = {}
    for entry in root.findall("atom:entry", NS):
        identifier = entry.findtext("atom:id", default="", namespaces=NS).rsplit("/abs/", 1)[-1]
        try:
            base_id, number, _ = version_output(identifier)
        except ValueError:
            continue  # Atom error-entry không phải paper metadata.
        if base_id not in requested_ids:
            continue
        # Benchmark chỉ cần độ phủ Version; bỏ title/authors và response thô.
        found[base_id] = {
            "arxiv_id": base_id, "latest_version": number,
        }
    return found


def discover_papers(context):
    if DISCOVERY_MODE == "precomputed_input":
        return
    pending = [value for value in context["manifest"]["base_ids"] if value not in context["papers"]]
    stats = context["http"]["discovery"]
    for round_number in range(MAX_RETRY_ROUNDS + 1):
        retry_ids, hint = [], 0.0
        for offset in range(0, len(pending), DISCOVERY_BATCH_SIZE):
            batch = pending[offset:offset + DISCOVERY_BATCH_SIZE]
            for base_id in batch:
                context["paper_started"].setdefault(base_id, time.monotonic())
            started = time.monotonic()
            if round_number:
                stats["retries"] += 1
            try:
                with arxiv_get(context, "https://export.arxiv.org/api/query", "discovery",
                               params={"id_list": ",".join(batch), "max_results": len(batch)}) as response:
                    response.raise_for_status()
                    found = parse_metadata(response.content, batch)
                if not found:
                    raise ET.ParseError("No paper metadata in Atom response")
                context["papers"].update(found)
                for base_id in batch:
                    context["discovery_errors"].pop(base_id, None)
                    if base_id not in found:
                        context["discovery_errors"][base_id] = "not_returned"
            except (requests.RequestException, ET.ParseError) as error:
                status = error.response.status_code if isinstance(error, requests.HTTPError) and error.response is not None else None
                count_error(stats, error, status)
                for base_id in batch:
                    context["discovery_errors"][base_id] = str(status) if status else type(error).__name__
                if status is None or status == 429 or status >= 500:
                    retry_ids.extend(batch)
                    hint = max(hint, retry_after_seconds(getattr(getattr(error, "response", None), "headers", {})))
            finally:
                duration = time.monotonic() - started
                for base_id in batch:
                    context["discovery_batch_seconds"][base_id] = context["discovery_batch_seconds"].get(base_id, 0) + duration
                write_json(DISCOVERY_STATUS_PATH, context["papers"])
                save_checkpoint(context)
        if not retry_ids or round_number == MAX_RETRY_ROUNDS:
            break
        stats["retry_rounds"] += 1
        delay = retry_delay(round_number + 1, hint)
        stats["backoff_seconds"] += delay
        time.sleep(delay)
        pending = retry_ids


## 5. Source: tách tải mạng và xử lý file
Downloader chạy trong thread điều phối; một file worker chạy song song qua bounded queue. Source tar/gzip/plain TeX/ZIP được xử lý thật, đo bytes trước/sau lọc rồi xóa. PDF/DOCX có record rỗng; HTML/định dạng lạ là lỗi. `metadata.json` giữ lịch sử `id/path/state`; path trỏ tới file tạm đã dọn, `success` chỉ có nghĩa tải HTTP thành công. Chỉ đánh dấu Version hoàn tất sau file worker xử lý xong. Source wall bao cả downloader và file worker; không cộng service durations để suy ra wall.

In [ ]:
def extract_source(version_id, archive_path, target):
    """Giữ nguyên bytes .tex/.bib; chỉ non-TeX đã nhận diện mới nhận empty."""
    base_id, number, _ = version_output(version_id)
    record = {"id": version_id, "base_id": base_id, "version": number,
              "kind": "unknown", "status": "error", "files": [],
              "bytes_downloaded": archive_path.stat().st_size, "bytes_before": 0, "bytes_after": 0}
    target.mkdir(parents=True, exist_ok=True)

    def safe_destination(name):
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("UnsafeArchivePath")
        destination = target / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        return destination

    try:
        with archive_path.open("rb") as source:
            signature = source.read(16)
        if tarfile.is_tarfile(archive_path):
            record["kind"] = "tar"
            with tarfile.open(archive_path, "r:*") as archive:
                for member in archive:
                    if not member.isfile():
                        continue
                    record["bytes_before"] += member.size
                    if Path(member.name).suffix.lower() in {".tex", ".bib"}:
                        destination = safe_destination(member.name)
                        with archive.extractfile(member) as source, destination.open("wb") as output:
                            shutil.copyfileobj(source, output)
                        record["files"].append(str(destination.relative_to(target)))
        elif zipfile.is_zipfile(archive_path):
            with zipfile.ZipFile(archive_path) as archive:
                record["bytes_before"] = sum(member.file_size for member in archive.infolist())
                record["kind"] = "docx" if "word/document.xml" in archive.namelist() else "zip"
                for member in archive.infolist():
                    if not member.is_dir() and Path(member.filename).suffix.lower() in {".tex", ".bib"}:
                        destination = safe_destination(member.filename)
                        with archive.open(member) as source, destination.open("wb") as output:
                            shutil.copyfileobj(source, output)
                        record["files"].append(str(destination.relative_to(target)))
        else:
            # GZIP có thể chứa một TeX hoặc PDF. Giải nén streaming rồi nhận diện.
            decoded = target / "payload.tmp"
            if signature.startswith(b"\x1f\x8b"):
                with gzip.open(archive_path, "rb") as source, decoded.open("wb") as output:
                    shutil.copyfileobj(source, output)
                payload = decoded
            else:
                payload = archive_path
            record["bytes_before"] = payload.stat().st_size
            with payload.open("rb") as source:
                prefix = source.read(8192).lstrip()
            if prefix.startswith(b"%PDF-"):
                record.update(kind="pdf", status="empty")
            elif prefix.startswith(b"{\\rtf") or prefix.startswith(b"\xd0\xcf\x11\xe0"):
                record.update(kind="document", status="empty")
            elif prefix.lower().startswith((b"<!doctype html", b"<html", b"<?xml")):
                record["error"] = "UnexpectedWebResponse"
            else:
                # Tìm xuyên suốt file, không bỏ TeX có phần comment dài ở đầu.
                markers = re.compile(rb"\\(?:documentclass|documentstyle|begin\s*\{document\}|input\b)")
                tex_found = False
                with payload.open("rb") as source:
                    tail = b""
                    for chunk in iter(lambda: source.read(MB), b""):
                        if b"\0" in chunk:
                            break
                        if markers.search(tail + chunk):
                            tex_found = True
                        tail = chunk[-128:]
                    else:
                        if tex_found:
                            shutil.copyfile(payload, target / "main.tex")
                            record.update(kind="gz_single" if payload == decoded else "tex_single", files=["main.tex"])
                if not record["files"]:
                    record["error"] = "UnsupportedSourceFormat"
            decoded.unlink(missing_ok=True)
        record["files"] = sorted(set(record["files"]))
        record["bytes_after"] = sum((target / name).stat().st_size for name in record["files"])
        if record["kind"] in {"tar", "zip", "docx", "gz_single", "tex_single"}:
            record["status"] = "ok" if any(Path(name).suffix.lower() == ".tex" for name in record["files"]) else "empty"
    except (tarfile.TarError, zipfile.BadZipFile, OSError, EOFError, zlib.error, ValueError) as error:
        record.update(status="error", error=type(error).__name__, files=[], bytes_after=0)
    return record


def cleanup_temp_items(items):
    for directory in items:
        if directory.exists():
            shutil.rmtree(directory)
    items.clear()


def crawl_sources(context):
    """Một producer tải mạng; một consumer xử lý file, queue bounded để overlap I/O."""
    from concurrent.futures import ThreadPoolExecutor

    stats = context["http"]["source"]
    pending = [value for value in context["manifest"]["selected_ids"]
               if context["statuses"].get(value, {}).get("status") not in TERMINAL_SOURCE]
    file_queue = queue.Queue(maxsize=FILE_QUEUE_SIZE)
    producer_done = threading.Event()
    stop = context["source_stop"]
    owned_directories = []

    def consume_files():
        cleanup_queue = []
        try:
            while not producer_done.is_set() or not file_queue.empty():
                try:
                    job = file_queue.get(timeout=0.1)
                except queue.Empty:
                    continue
                try:
                    if stop.is_set():
                        continue  # Không xử lý queue sau khi user ngắt; status downloaded sẽ được retry khi resume.
                    version_id, scratch, download_record, queued_at = job
                    started = time.monotonic()
                    queue_wait = started - queued_at
                    record = extract_source(version_id, scratch / "source.download", scratch / "extracted")
                    context["monitor"].sample_once(force_disk=True)
                    cleanup_queue.append(scratch)
                    if CLEANUP_MODE == "immediate" or len(cleanup_queue) >= BATCH_SIZE:
                        cleanup_temp_items(cleanup_queue)
                    elapsed = time.monotonic() - started
                    with context["state_lock"]:
                        if record["status"] == "error":
                            name = record.get("error", "ExtractionError")
                            stats["errors_by_type"][name] = stats["errors_by_type"].get(name, 0) + 1
                        record.update(seconds=download_record["seconds"] + elapsed,
                                      download_seconds=download_record["download_seconds"],
                                      file_processing_seconds=download_record.get("file_processing_seconds", 0) + elapsed,
                                      queue_wait_seconds=download_record.get("queue_wait_seconds", 0) + queue_wait,
                                      attempts=download_record["attempts"])
                        context["statuses"][version_id] = record
                        base_id = record["base_id"]
                        context["source_session_seconds"][base_id] += elapsed
                        context["paper_finished"][base_id] = time.monotonic()
                    print(f"[SOURCE] {version_id}: {record['status']}")
                finally:
                    file_queue.task_done()
        except BaseException:
            stop.set()  # Producer đang put queue đầy sẽ thoát thay vì deadlock.
            raise
        finally:
            cleanup_temp_items(cleanup_queue)

    try:
        with ThreadPoolExecutor(max_workers=1, thread_name_prefix="source-files") as executor:
            consumer = executor.submit(consume_files)
            try:
                for round_number in range(MAX_RETRY_ROUNDS + 1):
                    retry_ids, hint = [], 0.0
                    for index, version_id in enumerate(pending):
                        if stop.is_set():
                            break
                        if consumer.done():
                            consumer.result()
                        base_id, number, _ = version_output(version_id)
                        context["source_attempted_ids"].add(version_id)
                        context["paper_started"].setdefault(base_id, time.monotonic())
                        started = time.monotonic()
                        previous = context["statuses"].get(version_id, {})
                        scratch = Path(tempfile.mkdtemp(prefix="source-", dir=TEMP_DIR))
                        owned_directories.append(scratch)
                        record = {"id": version_id, "base_id": base_id, "version": number,
                                  "status": "error", "kind": "download_error", "files": [],
                                  "bytes_downloaded": 0, "bytes_before": 0, "bytes_after": 0}
                        state = "fail"
                        if round_number:
                            stats["retries"] += 1
                        try:
                            with arxiv_get(context, f"https://export.arxiv.org/src/{version_id}", "source", stream=True) as response:
                                response.raise_for_status()
                                with (scratch / "source.download").open("wb") as output:
                                    for chunk in response.iter_content(chunk_size=MB):
                                        output.write(chunk)
                            record.update(status="downloaded", kind="pending_processing")
                            state = "success"
                        except requests.RequestException as error:
                            status = error.response.status_code if isinstance(error, requests.HTTPError) and error.response is not None else None
                            with context["state_lock"]:
                                count_error(stats, error, status)
                            record["error"] = type(error).__name__
                            if status is not None:
                                record["http_status"] = status
                            if status == 429 or (status is not None and status >= 500) or isinstance(error, (
                                    requests.Timeout, requests.ConnectionError, requests.exceptions.ChunkedEncodingError,
                                    requests.exceptions.ContentDecodingError)):
                                retry_ids.append(version_id)
                                hint = max(hint, retry_after_seconds(getattr(getattr(error, "response", None), "headers", {})))
                        except OSError as error:
                            with context["state_lock"]:
                                count_error(stats, error)
                            record["error"] = type(error).__name__
                        finally:
                            elapsed = time.monotonic() - started
                            record.update(seconds=previous.get("seconds", 0) + elapsed,
                                          download_seconds=previous.get("download_seconds", 0) + elapsed,
                                          file_processing_seconds=previous.get("file_processing_seconds", 0),
                                          attempts=previous.get("attempts", 0) + 1)
                            with context["state_lock"]:
                                context["statuses"][version_id] = record
                                context["source_session_seconds"][base_id] = context["source_session_seconds"].get(base_id, 0) + elapsed
                                context["paper_finished"][base_id] = time.monotonic()
                                context["history"].append({"id": version_id, "path": str(scratch / "source.download"), "state": state})
                        if state == "success":
                            queued_at = time.monotonic()
                            while not stop.is_set():
                                if consumer.done():
                                    consumer.result()
                                try:
                                    file_queue.put((version_id, scratch, record, queued_at), timeout=0.1)
                                    break
                                except queue.Full:
                                    pass
                        else:
                            context["monitor"].sample_once(force_disk=True)
                            cleanup_temp_items([scratch])
                        if (index + 1) % BATCH_SIZE == 0:
                            save_checkpoint(context)
                    save_checkpoint(context)
                    if stop.is_set() or not retry_ids or round_number == MAX_RETRY_ROUNDS:
                        break
                    stats["retry_rounds"] += 1
                    delay = retry_delay(round_number + 1, hint)
                    stats["backoff_seconds"] += delay
                    if stop.wait(delay):
                        break
                    with context["state_lock"]:
                        for version_id in retry_ids:
                            context["statuses"][version_id]["seconds"] += delay
                            context["source_session_seconds"][version_output(version_id)[0]] += delay
                    pending = retry_ids
            finally:
                producer_done.set()
                # Không xóa scratch hoặc chốt benchmark khi file worker còn chạy.
                consumer.result()
    finally:
        cleanup_temp_items(owned_directories)
        save_checkpoint(context)


async def run_source_stage(context):
    from concurrent.futures import ThreadPoolExecutor
    executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="source-network")
    future = executor.submit(crawl_sources, context)
    try:
        # Poll hữu hạn giữ kernel nhận Stop trong khi network/file threads chạy.
        while not future.done():
            await asyncio.sleep(0.05)
        future.result()
    except (asyncio.CancelledError, KeyboardInterrupt):
        context["source_stop"].set()
        while not future.done():
            await asyncio.sleep(0.05)
        future.result()
        raise
    finally:
        executor.shutdown(wait=True)


## 6. Semantic Scholar
Một Session và worker async mỗi phần tử mảng key. Theo `next` lấy hết trang, đếm references có/không arXiv rồi bỏ page. Chỉ công bố counts thành công sau trang cuối. `[]` là thành công 0 references, khác với lỗi. Trung bình reference tính trên Publication S2 thành công, kể cả 0. Không lưu title/authors/raw references.

In [ ]:
async def fetch_s2(session, base_id, stats, worker_clock):
    """Đếm từng trang rồi bỏ response; chỉ công bố số đếm sau trang cuối."""
    offset = 0
    n_refs = n_arxiv = n_unresolved = 0
    url = f"https://api.semanticscholar.org/graph/v1/paper/ARXIV:{quote(base_id, safe='')}/references"
    while True:
        remaining = S2_REQUEST_INTERVAL - (time.monotonic() - worker_clock[0])
        if remaining > 0:
            await asyncio.sleep(remaining)
        worker_clock[0] = time.monotonic()
        stats["requests"] += 1  # Từng HTTP page, không chỉ mỗi Publication.
        params = {"fields": "paperId,title,authors,externalIds,year,publicationDate,venue",
                  "limit": 1000, "offset": offset}
        async with session.get(url, params=params) as response:
            response.raise_for_status()
            page = await response.json()
        for item in page["data"]:
            reference = item.get("citedPaper")
            if isinstance(reference, dict):
                n_refs += 1
                n_arxiv += bool((reference.get("externalIds") or {}).get("ArXiv"))
            else:
                n_unresolved += 1
        next_offset = page.get("next")
        if next_offset is None:
            break
        if next_offset <= offset:
            raise ValueError("NonIncreasingS2Offset")
        offset = next_offset
    return {"n_refs": n_refs, "n_refs_with_arxiv_id": n_arxiv, "n_unresolved_references": n_unresolved}


async def crawl_s2(context):
    from contextlib import AsyncExitStack
    stats = context["http"]["references"]
    pending = [base_id for base_id in context["manifest"]["reference_ids"]
               if context["s2_statuses"].get(base_id, {}).get("status") != "ok"]
    clocks = [[-float("inf")] for _key in S2_API_KEYS]
    async with AsyncExitStack() as stack:
        sessions = [await stack.enter_async_context(aiohttp.ClientSession(
            headers={"x-api-key": key}, timeout=aiohttp.ClientTimeout(total=S2_REQUEST_TIMEOUT)))
            for key in S2_API_KEYS]
        for round_number in range(MAX_RETRY_ROUNDS + 1):
            retry_ids = []
            hints = []

            async def worker(index, assigned_ids):
                for base_id in assigned_ids:
                    context["paper_started"].setdefault(base_id, time.monotonic())
                    started = time.monotonic()
                    previous = context["s2_statuses"].get(base_id, {})
                    record = {"status": "error"}
                    if round_number:
                        stats["retries"] += 1
                    try:
                        data = await fetch_s2(sessions[index], base_id, stats, clocks[index])
                        record = {"status": "ok", **data}
                    except aiohttp.ClientResponseError as error:
                        count_error(stats, error, error.status)
                        record.update(error=type(error).__name__, http_status=error.status)
                        if error.status == 429 or error.status >= 500:
                            retry_ids.append(base_id)
                            hints.append(retry_after_seconds(error.headers))
                    except (aiohttp.ClientError, asyncio.TimeoutError) as error:
                        count_error(stats, error)
                        record["error"] = type(error).__name__
                        retry_ids.append(base_id)
                    except (ValueError, KeyError) as error:
                        count_error(stats, error)
                        record["error"] = type(error).__name__
                    finally:
                        elapsed = time.monotonic() - started
                        record.update(seconds=previous.get("seconds", 0) + elapsed,
                                      attempts=previous.get("attempts", 0) + 1)
                        context["s2_statuses"][base_id] = record
                        context["s2_session_seconds"][base_id] = context["s2_session_seconds"].get(base_id, 0) + elapsed
                        context["paper_finished"][base_id] = time.monotonic()
                    print(f"[S2] {base_id}: {record['status']}")

            for start in range(0, len(pending), S2_BATCH_SIZE):
                batch = pending[start:start + S2_BATCH_SIZE]
                tasks = [asyncio.create_task(worker(index, batch[index::len(sessions)])) for index in range(len(sessions))]
                try:
                    await asyncio.gather(*tasks)
                finally:
                    for task in tasks:
                        if not task.done():
                            task.cancel()
                    await asyncio.gather(*tasks, return_exceptions=True)
                    context["monitor"].sample_once(force_disk=True)
                    save_checkpoint(context)
            if not retry_ids or round_number == MAX_RETRY_ROUNDS:
                break
            stats["retry_rounds"] += 1
            delay = retry_delay(round_number + 1, max(hints, default=0))
            stats["backoff_seconds"] += delay
            await asyncio.sleep(delay)
            for base_id in retry_ids:
                context["s2_statuses"][base_id]["seconds"] += delay
                context["s2_session_seconds"][base_id] += delay
            pending = retry_ids


## 7. Checkpoint và thống kê
Giữ 10 JSON benchmark cũ, thêm `discovery_status.json`, `s2_status.json`, `checkpoint.json`. Tất cả output là JSON đo lường; không xuất corpus, raw metadata, Parquet hoặc thay schema của nhóm. Counts và processing time tích lũy từ checkpoint; HTTP/RAM/disk/wall thuộc phiên hiện tại. `bytes_after` là kích thước source sau lọc đã đo rồi xóa, khác với `final_disk_mb` chỉ gồm báo cáo/checkpoint.

In [ ]:
def save_checkpoint(context):
    with context["state_lock"]:
        write_json(STATUS_OUTPUT_PATH, context["statuses"])
        write_json(S2_STATUS_PATH, context["s2_statuses"])
        write_json(DISCOVERY_STATUS_PATH, context["papers"])
        write_json(METADATA_OUTPUT_DIR, context["history"])
        write_json(CHECKPOINT_PATH, {
            "schema_version": 1, "identity": context["manifest"]["identity"],
            "discovery_errors": context["discovery_errors"],
        })


def build_summary(context):
    manifest = context["manifest"]
    statuses, s2_statuses = context["statuses"], context["s2_statuses"]
    rows, done = [], {}
    for base_id in manifest["base_ids"]:
        expected = manifest["expected"][base_id]
        records = [statuses[value] for value in expected.values() if value in statuses]
        source_done = len(records) == len(expected) and all(record["status"] in TERMINAL_SOURCE for record in records)
        unavailable = any(record.get("http_status") == 404 for record in records)
        failed = any(record["status"] == "error" and record.get("http_status") != 404 for record in records)
        state = "done" if source_done else "not_attempted" if not records else "error" if failed else "unavailable" if unavailable else "incomplete"
        metadata = context["papers"].get(base_id)
        missing_versions = ([number for number in range(1, metadata["latest_version"] + 1) if number not in expected]
                            if metadata else None)
        s2 = s2_statuses.get(base_id, {})
        s2_ok = s2.get("status") == "ok"
        source_seconds = sum(record.get("seconds", 0) for record in records)
        session_seconds = context["source_session_seconds"].get(base_id, 0) + context["s2_session_seconds"].get(base_id, 0)
        start, end = context["paper_started"].get(base_id), context["paper_finished"].get(base_id)
        row = {
            "id": base_id, "status": state, "n_versions": len(expected),
            "versions_with_tex": sum(record["status"] == "ok" for record in records),
            "versions_non_tex": sum(record["status"] == "empty" for record in records),
            "versions_unavailable": sum(record.get("http_status") == 404 for record in records),
            "bytes_downloaded": sum(record.get("bytes_downloaded", 0) for record in records),
            "bytes_before": sum(record.get("bytes_before", 0) for record in records if record["status"] in TERMINAL_SOURCE),
            "bytes_after": sum(record.get("bytes_after", 0) for record in records if record["status"] in TERMINAL_SOURCE),
            "seconds": source_seconds,  # Giữ nghĩa cũ: source, tất cả attempts đã lưu.
            "source_seconds": source_seconds, "references_seconds": s2.get("seconds"),
            "processing_seconds": source_seconds + s2.get("seconds", 0),
            "session_processing_seconds": session_seconds,
            "session_turnaround_seconds": end - start if start is not None and end is not None else None,
            "discovery_batch_seconds": context["discovery_batch_seconds"].get(base_id),
            "s2_status": "ok" if s2_ok else s2.get("status", "not_attempted"),
            "n_refs": s2.get("n_refs") if s2_ok else None,
            "n_refs_with_arxiv_id": s2.get("n_refs_with_arxiv_id") if s2_ok else None,
            "n_references": s2.get("n_refs_with_arxiv_id") if s2_ok else None,
            "missing_manifest_versions": missing_versions,
            "discovery_verified": metadata is not None and not missing_versions,
            "pipeline_complete": source_done and s2_ok and metadata is not None and not missing_versions,
        }
        rows.append(row)
        if source_done:
            done[base_id] = {"id": base_id, "status": "done", "n_versions": len(expected),
                             **{key: row[key] for key in ("versions_with_tex", "versions_non_tex", "bytes_downloaded",
                                                         "bytes_before", "bytes_after", "seconds")}}
    selected = [statuses[value] for value in manifest["selected_ids"] if value in statuses]
    sizes = [row for row in rows if row["status"] == "done"]
    refs_ok = [row for row in rows if row["s2_status"] == "ok"]
    before = sum(row["bytes_before"] for row in sizes)
    after = sum(row["bytes_after"] for row in sizes)
    states = Counter(row["status"] for row in rows)
    summary = {
        "papers_requested": len(rows), "papers_done": len(done),
        "papers_unavailable": states["unavailable"], "papers_error": states["error"],
        "papers_incomplete": states["incomplete"], "papers_not_attempted": states["not_attempted"],
        "paper_success_rate": ratio(len(done), len(rows)),
        "versions_requested": len(manifest["selected_ids"]), "versions_total": len(selected),
        "versions_expected_in_manifest": sum(len(manifest["expected"][value]) for value in manifest["base_ids"]),
        "versions_with_tex": sum(record["status"] == "ok" for record in selected),
        "versions_non_tex": sum(record["status"] == "empty" for record in selected),
        "versions_unavailable": sum(record.get("http_status") == 404 for record in selected),
        "versions_failed": sum(record["status"] == "error" for record in selected),
        "version_success_rate": ratio(sum(record["status"] in TERMINAL_SOURCE for record in selected), len(manifest["selected_ids"])),
        "version_tex_rate": ratio(sum(record["status"] == "ok" for record in selected), len(selected)),
        "total_bytes_downloaded": sum(record.get("bytes_downloaded", 0) for record in selected),
        "total_bytes_before": sum(record.get("bytes_before", 0) for record in selected if record["status"] in TERMINAL_SOURCE),
        "total_bytes_after": sum(record.get("bytes_after", 0) for record in selected if record["status"] in TERMINAL_SOURCE),
        "avg_bytes_downloaded_per_paper": average([row["bytes_downloaded"] for row in sizes]),
        "avg_bytes_before_per_paper": average([row["bytes_before"] for row in sizes]),
        "avg_bytes_after_per_paper": average([row["bytes_after"] for row in sizes]),
        "papers_with_size_before": len(sizes),
        "size_reduction_pct": 100 * (1 - after / before) if before else None,
        "avg_refs_per_paper": average([row["n_refs"] for row in refs_ok]),
        "avg_refs_with_arxiv_id_per_paper": average([row["n_refs_with_arxiv_id"] for row in refs_ok]),
        "avg_references_per_paper": average([row["n_refs_with_arxiv_id"] for row in refs_ok]),
        "references_total": sum(row["n_refs"] for row in refs_ok),
        "references_with_arxiv_id": sum(row["n_refs_with_arxiv_id"] for row in refs_ok),
        "papers_with_ref_metadata": len(refs_ok),
        "papers_requested_s2": len(manifest["reference_ids"]),
        "papers_s2_failed": sum(s2_statuses.get(value, {}).get("status") == "error" for value in manifest["reference_ids"]),
        "papers_s2_empty": sum(row["n_refs"] == 0 for row in refs_ok),
        "ref_metadata_rate": ratio(len(refs_ok), len(manifest["reference_ids"])),
        "pipeline_complete": sum(row["pipeline_complete"] for row in rows),
        "pipeline_success_rate": ratio(sum(row["pipeline_complete"] for row in rows), len(rows)),
        "papers_discovery_verified": sum(row["discovery_verified"] for row in rows),
        "avg_processing_seconds_per_paper": average([row["processing_seconds"] for row in rows]),
        "avg_session_processing_seconds_per_paper": average([row["session_processing_seconds"] for row in rows]),
        "avg_session_turnaround_seconds_per_paper": average([row["session_turnaround_seconds"] for row in rows]),
        "source_versions_processed_this_session": len(context["source_attempted_ids"]),
        "reference_denominator": "Publications with successful complete S2 responses, including zero references",
        "size_denominator": "Publications source-complete in the input manifest; same set before and after filtering",
        "counts_scope": "checkpoint + current session; HTTP and benchmark are current session only",
    }
    return summary, rows, done


def total_http_stats(context):
    total = new_http_stats()
    for stats in context["http"].values():
        for key in total:
            if key != "errors_by_type":
                total[key] += stats[key]
        total["errors_by_type"] = dict(Counter(total["errors_by_type"]) + Counter(stats["errors_by_type"]))
    return total


def persist_reports(context, run_state):
    summary, rows, done = build_summary(context)
    benchmark = context["monitor"].summary(context["done_at_start"], len(done))
    benchmark["http"] = total_http_stats(context)
    summary["avg_wall_seconds_per_requested_paper"] = benchmark["wall_seconds"] / len(rows)
    stages = {name: {"seconds": context["stage_seconds"][name], **context["http"][name]}
              for name in ("discovery", "source", "references")}
    stages["export"] = {"seconds": context["stage_seconds"]["export"]}
    scope_complete = (DISCOVERY_MODE == "live" and context["cold_run"] and run_state == "completed"
                      and set(context["manifest"]["reference_ids"]) == set(context["manifest"]["base_ids"]))
    claim = {
        "schema_version": 1, "member": MEMBER,
        "range": {"segments": segments_for(context["manifest"]["base_ids"])},
        "ids_attempted": len(rows),
        "run": {"cold_run": context["cold_run"], "resumed_passes": int(not context["cold_run"]),
                "started_at": benchmark["started_iso"], "finished_at": benchmark["finished_iso"],
                "machine": context["machine"]["platform"], "python": context["machine"]["python"],
                "s2_api_key": True, "s2_workers": len(S2_API_KEYS),
                "measured_by": "BenchmarkMonitor; time-weighted RSS process tree and non-overlapping disk roots"},
        "stages": stages,
        "totals": {
            "wall_seconds": benchmark["wall_seconds"], "publications_found": len(context["papers"]),
            "publications_complete": summary["pipeline_complete"],
            "versions_expected": summary["versions_requested"],
            "versions_ok": summary["versions_with_tex"] + summary["versions_non_tex"],
            "versions_with_tex": summary["versions_with_tex"], "versions_non_tex": summary["versions_non_tex"],
            "versions_failed": summary["versions_failed"],
            "references_total": summary["references_total"], "references_with_arxiv_id": summary["references_with_arxiv_id"],
            "publications_s2_ok": summary["papers_with_ref_metadata"],
            "bytes_downloaded": summary["total_bytes_downloaded"],
            "bytes_before_figure_removal": summary["total_bytes_before"],
            "bytes_after_figure_removal": summary["total_bytes_after"],
            "size_reduction_pct": summary["size_reduction_pct"],
            "peak_rss_mb": benchmark["peak_rss_mb"], "avg_rss_mb": benchmark["avg_rss_mb"],
            "peak_disk_mb": benchmark["peak_disk_mb"], "output_mb": benchmark["final_disk_mb"],
        },
        "http": benchmark["http"], "end_to_end_complete": scope_complete,
        "notes": "Benchmark only, no corpus/metadata payload retained. Boundary: existing Version manifest -> live "
                 "metadata resolution -> source download + file processing -> S2 -> first report export. "
                 "Manifest generation and final telemetry rewrite excluded. Resume metrics cover this session only.",
    }
    run = {
        "session": context["session"], "member": MEMBER, "state": run_state,
        "slice": {"start_idx": START_IDX, "end_idx": END_IDX, "versions_count": len(context["manifest"]["selected_ids"])},
        "identity": context["manifest"]["identity"], "machine": context["machine"],
        "cold_run": context["cold_run"], "benchmark_only": True, "payload_retained": False,
        "cleanup_mode": CLEANUP_MODE, "file_queue_size": FILE_QUEUE_SIZE,
        "source_downloader_workers": 1, "source_file_workers": 1,
        "s2_workers": len(S2_API_KEYS), "discovery_mode": DISCOVERY_MODE,
        "discovery_errors": context["discovery_errors"], "stages": stages,
        "end_to_end_complete": scope_complete, "benchmark": benchmark, "summary": summary,
        "duplicate_selected_lines": context["manifest"]["duplicate_selected_lines"],
        "reference_ids_outside_slice": context["manifest"]["reference_ids_outside_slice"],
        "timing_definitions": {
            "wall_seconds": "This session, all measured stages; never sum parallel worker durations",
            "papers_stats.seconds": "Source processing across stored attempts; legacy metric",
            "processing_seconds": "Source + S2 processing and attributed retry waits across stored attempts; excludes discovery",
            "session_turnaround_seconds": "Observed time from first work on a paper to last source/S2 completion, including queueing",
            "discovery_batch_seconds": "Shared observed batch duration; do not sum across papers",
            "source_stage_seconds": "Wall time with network and file processing overlapped; do not sum service durations",
            "bytes_after": "Size of filtered source measured temporarily and discarded; not final retained output",
        },
    }
    save_checkpoint(context)
    write_json(DONE_OUTPUT_PATH, done)
    write_json(SKIPPED_OUTPUT_PATH, [value for value, record in context["statuses"].items() if record["status"] != "ok"])
    write_json(PAPERS_STATS_OUTPUT_PATH, rows)
    write_json(SUMMARY_OUTPUT_PATH, summary)
    write_json(BENCHMARK_SAMPLES_OUTPUT_PATH, context["monitor"].samples.copy())
    write_json(BENCHMARK_OUTPUT_PATH, benchmark)
    write_json(CLAIM_OUTPUT_PATH, claim)
    write_json(RUN_OUTPUT_PATH, run)
    return {"summary": summary, "papers": rows, "benchmark": benchmark, "claim": claim, "run": run}


## 8. Điều phối
Discovery → source pipeline (mạng/file overlap) → S2 nhiều worker → export. Monitor bao tất cả stage. Retry giới hạn cho 429/5xx và lỗi truyền tải, backoff/Retry-After. Khi ngắt, signal dừng và chờ thread đang chạy kết thúc trước cleanup/final checkpoint; timeout request hữu hạn, nên Stop có thể phải chờ request đang chạy. Status `downloaded` còn trong queue chưa được tính thành công và sẽ tải lại khi resume. Checkpoint source mỗi `BATCH_SIZE`, S2 mỗi `S2_BATCH_SIZE`; mất runtime đột ngột có thể phải làm lại batch dở. Chọn output trên mount bền vững nếu cần sống qua reset Colab; notebook không tự mount/upload.

In [ ]:
async def run_pipeline():
    global COLD_RUN
    if not S2_API_KEYS or any(not isinstance(key, str) or not key.strip() or key.startswith("PASTE_") for key in S2_API_KEYS):
        raise ValueError("Điền S2_API_KEYS trong cell cấu hình; mỗi phần tử là một key/worker.")
    if not CONTACT_EMAIL:
        raise ValueError("Điền CONTACT_EMAIL trước khi gọi arXiv.")
    if DISCOVERY_MODE not in {"live", "precomputed_input"} or CLEANUP_MODE not in {"immediate", "batch"}:
        raise ValueError("Kiểm tra DISCOVERY_MODE và CLEANUP_MODE")
    if min(BATCH_SIZE, S2_BATCH_SIZE, DISCOVERY_BATCH_SIZE, FILE_QUEUE_SIZE, BENCHMARK_INTERVAL, DISK_INTERVAL) <= 0 or MAX_RETRY_ROUNDS < 0:
        raise ValueError("Batch/interval phải dương và MAX_RETRY_ROUNDS >= 0")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)
    monitor = BenchmarkMonitor(OUTPUT_ROOT, TEMP_DIR, BENCHMARK_INTERVAL, DISK_INTERVAL)
    monitor.start()
    context = None
    run_state = "error"
    try:
        manifest = load_manifest()
        checkpoint = read_json(CHECKPOINT_PATH, None)
        if checkpoint and (not RESUME or checkpoint["identity"] != manifest["identity"]):
            raise ValueError("Output đã có checkpoint khác input/chế độ. Chọn OUTPUT_ROOT mới để giữ dữ liệu cũ.")
        if not checkpoint and any(path.exists() for path in (STATUS_OUTPUT_PATH, S2_STATUS_PATH, DISCOVERY_STATUS_PATH)):
            raise ValueError("Output có dữ liệu nhưng thiếu checkpoint. Chọn OUTPUT_ROOT mới.")
        statuses = read_json(STATUS_OUTPUT_PATH, {}) if checkpoint else {}
        s2_statuses = read_json(S2_STATUS_PATH, {}) if checkpoint else {}
        papers = read_json(DISCOVERY_STATUS_PATH, {}) if checkpoint else {}
        COLD_RUN = not bool(statuses or s2_statuses or papers)
        session = requests.Session()
        session.headers.update({"User-Agent": USER_AGENT})
        context = {
            "manifest": manifest, "monitor": monitor, "machine": get_machine_info(),
            "session": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%S.%fZ"),
            "cold_run": COLD_RUN, "arxiv_session": session, "last_arxiv_request": -float("inf"),
            "statuses": statuses, "s2_statuses": s2_statuses, "papers": papers,
            "history": read_json(METADATA_OUTPUT_DIR, []) if checkpoint else [],
            "discovery_errors": checkpoint.get("discovery_errors", {}) if checkpoint else {},
            "http": {name: new_http_stats() for name in ("discovery", "source", "references")},
            "stage_seconds": {"discovery": None if DISCOVERY_MODE == "precomputed_input" else 0.0,
                              "source": 0.0, "references": 0.0, "export": 0.0},
            "source_session_seconds": {}, "s2_session_seconds": {}, "discovery_batch_seconds": {},
            "paper_started": {}, "paper_finished": {}, "source_attempted_ids": set(), "done_at_start": 0,
            "state_lock": threading.RLock(), "source_stop": threading.Event(),
        }
        context["done_at_start"] = len(build_summary(context)[2])
        print(f"[START] {len(manifest['selected_ids'])} Versions; {len(manifest['base_ids'])} Publications; "
              f"S2 workers={len(S2_API_KEYS)}; cold_run={COLD_RUN}")
        if DISCOVERY_MODE == "live":
            started = time.monotonic()
            try:
                discover_papers(context)
            finally:
                context["stage_seconds"]["discovery"] = time.monotonic() - started
        started = time.monotonic()
        try:
            await run_source_stage(context)
        finally:
            context["stage_seconds"]["source"] = time.monotonic() - started
        started = time.monotonic()
        try:
            await crawl_s2(context)
        finally:
            context["stage_seconds"]["references"] = time.monotonic() - started
        run_state = "completed"
    except (KeyboardInterrupt, asyncio.CancelledError):
        run_state = "interrupted"
        raise
    finally:
        if context is None:
            monitor.stop()
        else:
            context["arxiv_session"].close()
            started = time.monotonic()
            try:
                # Export lần đầu vẫn ở trong vùng đo RAM/disk/wall của toàn pipeline.
                persist_reports(context, run_state)
            finally:
                context["stage_seconds"]["export"] = time.monotonic() - started
                monitor.stop()
            # Ghi số đo đã đóng băng. Lần rewrite telemetry này nằm ngoài vùng đo.
            # Cập nhật final disk đến khi kích thước các JSON tự chứa metric ổn định.
            for _ in range(5):
                results = persist_reports(context, run_state)
                actual_mb = dir_size(OUTPUT_ROOT) / MB
                if actual_mb == results["benchmark"]["final_disk_mb"]:
                    break
    return results


## 9. Chạy toàn pipeline
Cell này thực hiện request thật khi bạn chạy. Bộ kiểm thử cục bộ dùng mock, không gọi arXiv/S2.

In [ ]:
# Top-level await được Jupyter/Colab hỗ trợ. Không gọi asyncio.run() trong notebook.
results = await run_pipeline()


## 10. Kết quả
Bảng được sinh từ JSON bằng code; dùng JSON cho Jupyter Book, không crawl lại khi build báo cáo.

- `wall_seconds / papers_requested`: wall trung bình của toàn run, kể cả các phần song song.
- `papers_stats.seconds`: thời gian service source tích lũy (download + file processing + retry waits), không phải source wall.
- `processing_seconds`: source + S2 service, chưa gồm discovery; không cộng thành wall.
- `session_turnaround_seconds`: từ lúc bắt đầu Publication đến khi kết thúc trong phiên, gồm chờ stage/queue. `discovery_batch_seconds` dùng chung cho cả batch, không cộng theo paper.
- `bytes_after`: dung lượng source lọc đã đo rồi bỏ; output cuối chỉ giữ JSON.
- `end_to_end_complete`: đủ stage trong scope của cold run, không bao thời gian tạo manifest ngoài notebook và không có nghĩa đã tạo dataset. Xem thêm `pipeline_success_rate` và missing Versions.
- Peak RAM/disk là giá trị quan sát theo chu kỳ, có lấy mẫu disk trước cleanup. Bước rewrite telemetry cuối nằm ngoài wall time đã đóng băng.

In [ ]:
from IPython.display import Markdown, display

summary = read_json(SUMMARY_OUTPUT_PATH, {})
benchmark = read_json(BENCHMARK_OUTPUT_PATH, {})
run = read_json(RUN_OUTPUT_PATH, {})
display(Markdown("| Chỉ số | Giá trị |\n|---|---|\n" + "\n".join(
    f"| {label} | {value} |" for label, value in [
        ("Trạng thái run", run.get("state")),
        ("Publication source hoàn tất / yêu cầu", f"{summary.get('papers_done')} / {summary.get('papers_requested')}"),
        ("Publication toàn pipeline hoàn tất", summary.get("pipeline_complete")),
        ("Tỉ lệ metadata S2 thành công", summary.get("ref_metadata_rate")),
        ("References trung bình / Publication S2 thành công", summary.get("avg_refs_per_paper")),
        ("Wall time cả lượt (giây)", benchmark.get("wall_seconds")),
        ("Wall time / Publication yêu cầu (giây)", summary.get("avg_wall_seconds_per_requested_paper")),
        ("RSS tối đa / trung bình (MiB)", f"{benchmark.get('peak_rss_mb')} / {benchmark.get('avg_rss_mb')}"),
        ("Disk đỉnh / output cuối (MiB)", f"{benchmark.get('peak_disk_mb')} / {benchmark.get('final_disk_mb')}"),
        ("Đủ scope discovery + source + S2 trong một cold run", run.get("end_to_end_complete")),
    ])))
print(f"Output: {OUTPUT_ROOT.resolve()}")
print("Thời gian các stage:", {name: value["seconds"] for name, value in run.get("stages", {}).items()})
